# Gridfall: reproduce it on free compute

This notebook rebuilds part of Hurricane Ian from NASA Black Marble and EAGLE-I on a Google Colab session and compares the result with what the repository committed. It needs nothing but a free NASA Earthdata token.

| Level | What it does | Time on Colab's free CPU runtime | Download |
|---|---|---|---|
| 0 | Runs the test suite and the sample export; checks the frozen lockbox record | 2-3 min (estimate) | none |
| 1 | Rebuilds Lee, Charlotte and Collier counties for all 55 Ian nights and compares every column with the committed sample | about 4.5 min (one run) | about 5.5 GB (satellite tiles stream through memory; the EAGLE-I file is 1.2 GB) |
| 2 | Rebuilds all of Ian (505 counties, 3 tiles) and compares it with a committed fingerprint | about 5.5 min (one run, right after level 1 in the same session) | about 14 GB |

Levels 1 and 2 were each timed once on Colab, on 2026-11-11 (US Pacific time), from the start of the run to the verdict; the time of level 0 is an estimate. The full Ian build alone took 8 minutes on an 8-core laptop.

**Before you run it**
1. Make a free account at https://urs.earthdata.nasa.gov and generate a token (Profile → Generate Token). Tokens expire 60 days after they are generated: make a new one when the token check fails.
2. In Colab, open the key icon on the left (Secrets), add `EARTHDATA_TOKEN` with that value, and allow this notebook to read it. Level 0 needs no token.
3. The repository is public, so the code downloads without a GitHub token. Only if you keep a private copy: set `REPO` in cell 1 to it and add a GitHub token as the secret `GITHUB_TOKEN` (read-only access to that copy is enough).
4. Keep the browser tab open: free Colab ends the session on disconnect and discards everything, so a dropped level-2 run starts again from zero.

The tokens are read from Colab's secret store into the process environment and are never printed, written to disk or put into a URL.

In [ ]:
#@title 1. Settings { display-mode: "form" }
LEVEL = 1  #@param [0, 1, 2] {type:"raw"}
REPO = "mukuna-web/gridfall"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import os

from google.colab import userdata


def secret(name, required):
    """A Colab secret, stripped of stray whitespace; None when it is missing or empty and not required."""
    try:
        value = userdata.get(name).strip()
        problem = None if value else f"The secret {name} is empty: paste the token into it and run this cell again"
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError) as err:
        problem = (f"Add the secret {name} in the key panel on the left and allow this notebook to read it "
                   f"({type(err).__name__})")
    except userdata.TimeoutException:  # the access prompt went unanswered
        problem = (f"Colab did not hand over the secret {name}: answer the access prompt (or switch on notebook "
                   f"access for {name} in the key panel on the left) and run this cell again")
    if problem is None:
        return value
    if required:
        raise SystemExit(problem)
    print(f"{name}: not set or not readable, continuing without it")
    return None


earthdata_token = secret("EARTHDATA_TOKEN", required=LEVEL >= 1)
if earthdata_token:
    os.environ["EARTHDATA_TOKEN"] = earthdata_token
github_token = secret("GITHUB_TOKEN", required=False)
del earthdata_token
print(f"level {LEVEL}; Earthdata token: {'loaded' if os.environ.get('EARTHDATA_TOKEN') else 'none'}; "
      f"GitHub token: {'loaded' if github_token else 'none (public clone)'}")

In [ ]:
#@title 2. Get the code and the Python environment (about 15 seconds when timed)
import os
import subprocess
import sys
from pathlib import Path

# Start in /content, wherever the kernel was left: after `!rm -rf` of the checkout (what this cell asks for when
# REPO or BRANCH changed) the kernel is still inside that folder, which is gone, and git cannot run from there.
os.chdir("/content")
DEST = Path("/content/gridfall")
SOURCE = f"{REPO} at {BRANCH}"
CLONED_FROM = DEST / ".git" / "cloned-from"  # written after the clone: what the checkout in DEST came from
# git asks this helper for the credential, so the token is never in the command line, the URL or .git/config.
HELPER = "!f() { echo username=x-access-token; echo \"password=$GIT_TOKEN\"; }; f"

if DEST.exists():
    found = CLONED_FROM.read_text() if CLONED_FROM.exists() else "some other checkout"
    if found != SOURCE:  # REPO or BRANCH changed since the clone: the cells below would report on the old code
        raise SystemExit(f"{DEST} holds {found}, not {SOURCE}: delete it (run `!rm -rf {DEST}` in a new cell) and "
                         f"run this cell again")
else:
    env = {**os.environ, "GIT_TOKEN": github_token or ""}
    auth = ["-c", f"credential.helper={HELPER}"] if github_token else []
    url = f"https://github.com/{REPO}.git"
    clone = subprocess.run(["git", *auth, "clone", "--quiet", "--branch", BRANCH, url, str(DEST)],
                           env=env, capture_output=True, text=True, check=False)
    if clone.returncode:
        raise SystemExit("git clone failed (check REPO, BRANCH and, for a private repository, GITHUB_TOKEN):\n"
                         + clone.stderr.strip())
    CLONED_FROM.write_text(SOURCE)
github_token = None
%cd /content/gridfall
!git remote -v
!git log --oneline -1
for cmd in ([sys.executable, "-m", "pip", "install", "-q", "uv>=0.11"],
            ["uv", "sync", "--locked"],
            ["uv", "run", "python", "--version"]):
    subprocess.run(cmd, cwd=DEST, check=True)  # stops the cell on the first failure instead of carrying on

In [ ]:
#@title 3. Reproduce
# An older report must never be read as this run's (the script also replaces it the moment it starts).
!rm -f /content/reproduce-report.json
!uv run python -I scripts/reproduce.py --level {LEVEL} --report /content/reproduce-report.json

In [ ]:
#@title 4. The report
import json
import os

path = "/content/reproduce-report.json"
if not os.path.exists(path):
    raise SystemExit("no report was written, so the run stopped before it started: read the output of cell 3")
with open(path) as fh:
    report = json.load(fh)
if report["level"] != LEVEL:
    raise SystemExit(f"the report is from a level {report['level']} run, not level {LEVEL}: run cell 3 again")
if report.get("complete") and report["passed"]:
    verdict = "PASS"
elif not all(check["passed"] for check in report["checks"]):
    verdict = "FAIL"
else:  # stopped, disconnected or killed before its last check: the checks so far say nothing about the rest
    verdict = "INCOMPLETE (the run did not reach its end, so there is no verdict: run cell 3 again)"
print(verdict, "- level", report["level"], "on", report["platform"])
for check in report["checks"]:
    print(f"  [{'PASS' if check['passed'] else 'FAIL'}] {check['name']} ({check['seconds']} s): {check['detail']}")
build = report.get("build") or {}
if build.get("problems"):
    print("satellite files that could not be read (a failed NASA download, so nothing was compared; run cell 3 again):")
    for problem in build["problems"]:
        print("   ", problem)

## What a pass means

- **Level 1:** the pipeline, run from scratch on a different machine, gives the same county-night table as the sample committed in the repository: every column of it (clear-pixel darkening, visibility, the EAGLE-I count at the satellite pass, customers, wind band, pass time and the rest), row by row, within floating-point tolerance. The sample was cut from the table that fed the recorded lockbox run.
- **Level 2:** the same for all 505 counties and 55 nights of Ian, compared against a committed fingerprint: counts and sums of every measured column for each night and for each county, the number of values present in each, the per-county inputs (wind band, customers, housing units, coverage) and Lee County's whole series. A fingerprint is a summary, not the table: it catches a changed value and a value that moved to another county, but it would miss a rearrangement that keeps every night's and every county's sums and counts.

**PASS** is printed only for a run that reached its end with every check passing. A run that was stopped, lost its session or died part-way is shown as **INCOMPLETE**, never as a pass.

The EAGLE-I, Census and HURDAT2 downloads are pinned in `docs/reproduce/inputs.json` (figshare MD5s and the EAGLE-I row count, Census and HURDAT2 SHA-256s), so a change in one of them is reported by its own check. NASA's satellite tiles are not pinned: each download is checked only against the MD5 NASA lists for it that day, so if NASA reprocesses the tiles, the comparison fails with different numbers, just as it would after a code change. A tile NASA could not serve is another matter: it is listed under **satellite downloads**, that check fails, and the comparison is not run on half the data but marked "not compared". Run cell 3 again.

The numbers in `docs/scorecard.md` follow from those tables, so reproducing the tables reproduces the scorecard's inputs. The one-shot lockbox result itself is frozen in `docs/lockbox_result.json` and is never recomputed here (decision log, D-12).